In [1]:
from pytential.sympy_pytential import sympy_pytential
import numpy as np
import matplotlib.pyplot as plt
from pytential.reduce.matrix_methods import reduce_qp

Create two ideal mixing functions from a set of properties, and check them.

In [2]:
from sympy import log, symbols
c0a, c0b, c0c, c0d, c1d, c0, c1, Va, Vb, Vc, Vd = symbols('c0a, c0b, c0c, c0d, c1d, c0, c1, Va, Vb, Vc, Vd')

In [3]:
T = 1600
RT = 8.134*T
mu0_SiC = -161028
rho_SiC = 3.21 / 40.11 * 1e6
rho_Ar = 101e3 / RT
v_SiC = 1/rho_SiC
v_Ar = 1/rho_Ar

In [4]:
fa_sp = c0a*mu0_SiC 
fb_sp = c0b*mu0_SiC
fc_sp = c0c*mu0_SiC 
fd_sp = c0d*(-120100+RT*log(c0d/(c0d+c1d))) +c1d*(-290457+RT*log(c1d/(c0d+c1d)))

In [5]:
np.exp((mu0_SiC--120100)/RT)

0.04307449610427493

In [6]:
fa = sympy_pytential(fa_sp, constraints_sym=[c0a*v_SiC-Va])
fb = sympy_pytential(fb_sp, constraints_sym=[c0b*v_SiC-Vb])
fc = sympy_pytential(fc_sp, constraints_sym=[c0c*v_SiC-Vc])
fd = sympy_pytential(fd_sp, constraints_sym=[c0d*v_Ar+c1d*v_Ar-Vd])

Make a function fa+fb with all the variables, and add constraints that the concentrations must sum to ca and cb

In [7]:
f = fa+fb+fc+fd
f = f.add_constraints_sym([c0a+c0b+c0c+c0d-c0, c1d-c1])
print(f)


Variables
['Va', 'Vb', 'Vc', 'Vd', 'c0', 'c0a', 'c0b', 'c0c', 'c0d', 'c1', 'c1d']

Potential
-161028*c0a - 161028*c0b - 161028*c0c + c0d*(13014.4*log(c0d/(c0d + c1d)) - 120100) + c1d*(13014.4*log(c1d/(c0d + c1d)) - 290457)

Gradient
[0, 0, 0, 0, 0, -161028, -161028, -161028, 13014.4*log(c0d/(c0d + c1d)) - 120100.0, 0, 13014.4*log(c1d/(c0d + c1d)) - 290457.0]

Hessian
[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 13014.4*c1d/(c0d*(c0d + c1d)), 0, -13014.4/(c0d + c1d)], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, -13014.4/(c0d + c1d), 0, 13014.4*c0d/(c1d*(c0d + c1d))]]

Constraints
-Va + 1.24953271028037e-5*c0a
-Vb + 1.24953271028037e-5*c0b
-Vc + 1.24953271028037e-5*c0c
-Vd + 0.128855445544554*c0d + 0.12885544554455

In [8]:
A = np.array(f.get_constraint_jacobian(), dtype = np.float64)
print(A.shape)
np.linalg.inv(A@A.T)

(6, 11)


array([[ 1.00000000e+00,  3.13281446e-11,  3.13281446e-11,
         3.15214336e-07, -2.50718884e-06, -2.03085419e-08],
       [ 3.13281446e-11,  1.00000000e+00,  3.13281446e-11,
         3.15214336e-07, -2.50718884e-06, -2.03085419e-08],
       [ 3.13281446e-11,  3.13281446e-11,  1.00000000e+00,
         3.15214336e-07, -2.50718884e-06, -2.03085419e-08],
       [ 3.15214336e-07,  3.15214336e-07,  3.15214336e-07,
         9.78871218e-01, -2.52265774e-02, -6.30664435e-02],
       [-2.50718884e-06, -2.50718884e-06, -2.50718884e-06,
        -2.52265774e-02,  2.00650116e-01,  1.62529093e-03],
       [-2.03085419e-08, -2.03085419e-08, -2.03085419e-08,
        -6.30664435e-02,  1.62529093e-03,  5.04063227e-01]])

In [9]:
y0 = {'Va':1, 'Vb':1, 'Vc':1, 'Vd':1, 'c0a':rho_SiC, 'c0b':rho_SiC, 'c0c':rho_SiC , 'c0d':rho_Ar*0.043, 'c1d':rho_Ar*(1-0.043), 'c0':3*rho_SiC+rho_Ar*.043,'c1':rho_Ar*(1-.043)}
print(y0)

{'Va': 1, 'Vb': 1, 'Vc': 1, 'Vd': 1, 'c0a': 80029.9177262528, 'c0b': 80029.9177262528, 'c0c': 80029.9177262528, 'c0d': 0.3337072780919596, 'c1d': 7.42692709613966, 'c0': 240090.08688603647, 'c1': 7.42692709613966}


In [10]:
fq = f.quadratic_expansion(y0)
print(fq)
print(fq.vars)
print(fq.grad(**y0))
print(fq.get_constraint_jacobian())



Variables
['Va', 'Vb', 'Vc', 'Vd', 'c0', 'c0a', 'c0b', 'c0c', 'c0d', 'c1', 'c1d']

Potential
-161028*c0a - 161028*c0b - 161028*c0c + 0.5*c0d*(37322.4727707852*c0d - 1676.97631049505*c1d) - 161050.527517103*c0d + 0.5*c1d*(-1676.97631049505*c0d + 75.3500327599657*c1d) - 291029.00744506*c1d - 38663387969.8234

Gradient
[0, 0, 0, 0, 0, -161028, -161028, -161028, 37322.4727707852*c0d - 1676.97631049505*c1d - 161050.527517103, 0, -1676.97631049505*c0d + 75.3500327599657*c1d - 291029.00744506]

Hessian
[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 37322.4727707852, 0, -1676.97631049505], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, -1676.97631049505, 0, 75.3500327599657]]

Constraints
-Va + 1.24953271028037e-5*c0a
-Vb + 1

In [12]:
print(fq.vars)
fr = fq.remove_linear_constraints(['c0', 'c1', 'Va', 'Vb', 'Vc', 'Vd'], y0=y0)
print(fr)
print(fr.vars)
print('grad', fr.grad(**y0))
y1 = {'c0': 80029.9177262528, 'c1': 0, 'Va': .9, 'Vb': 0, 'Vc': 0, 'Vd': 0}
print(fr(**y1))
print(fr.grad(**y1))

['Va', 'Vb', 'Vc', 'Vd', 'c0', 'c0a', 'c0b', 'c0c', 'c0d', 'c1', 'c1d']
free_indices [4, 9, 0, 1, 2, 3]
A_d has condition number 160143.58675631855, rank 5, and dimensions (6, 5)
Q_tilde is symmetric.
old function
 [[ 3.03303438e-08 -9.96131236e+00 -2.42375270e-03 -2.43253532e-03
  -2.43231265e-03  7.38866960e+01]
 [ 3.03672293e-08 -9.93883992e+00 -2.43162100e-03 -2.42745294e-03
  -2.42796485e-03  7.38485798e+01]
 [ 3.04240580e-08 -9.95854353e+00 -2.43708254e-03 -2.43246988e-03
  -2.43217941e-03  7.40230811e+01]
 [-9.23647678e-04  3.02660463e+05  7.39194437e+01  7.39194630e+01
   7.39194504e+01 -2.24783719e+06]
 [-3.79531527e-13  1.24364718e-04  3.03738759e-08  3.03738825e-08
   3.03738772e-08 -9.23647734e-04]
 [ 1.24364716e-04 -4.07517752e+04 -9.95289744e+00 -9.95290004e+00
  -9.95289834e+00  3.02660463e+05]] [1.76813742e+00 2.10118631e+00 1.76204248e+00 1.74828006e+02
 1.61028000e+05 2.91006480e+05]
sol [ 5.33536119e+04 -2.66763058e+04 -2.66763058e+04 -2.55015868e-04
  4.16499802e-06

In [ ]:
lam_lin = np.array([[-6.18428344e-24, -7.03045332e-07,  4.88411584e-18, -3.57416605e-18,
   5.22146643e-06],
 [ 1.16589051e-24,  1.32541448e-07, -9.20776737e-19,  6.73818775e-19,
  -9.84375673e-07],
 [ 2.18412391e-28,  2.48296854e-11, -1.72493941e-22,  1.26230009e-22,
  -1.84408263e-10],
 [ 2.66232918e-12,  3.02660465e+05, -2.10260805e-06,  1.53867569e-06,
  -2.24783721e+06],
 [ 6.17553483e-29,  7.02050768e-12, -4.87720652e-23,  3.56910985e-23,
  -5.21407987e-11],
 [-3.58469814e-13, -4.07517754e+04,  2.83106057e-07, -2.07175278e-07,
   3.02660465e+05]])
lam_const = [-6.47824565e-07, -4.89348047e-06,  6.71067203e-11,  1.74827824e+02,
  1.61028000e+05,  2.91006480e+05]
lam_lin@np.array([80029.9177262528, 0, 1, 0, 0])+lam_const

array([-6.47824565e-07, -4.89348047e-06,  6.71067203e-11,  1.74827822e+02,
        1.61028000e+05,  2.91006480e+05])

In [ ]:
[1.8783331795833331, -398267.9210385821, -95.39321080635419, 77.5576522980707, 160930.72964178462, 3248927.2350904713]

[1.8783331795833331,
 -398267.9210385821,
 -95.39321080635419,
 77.5576522980707,
 160930.72964178462,
 3248927.2350904713]